# Phase Template — copy this notebook to `notebooks/decisions/phase_<x>_<name>.ipynb`

Fixed shape every phase-decision notebook follows from Phase A onward:

1. Hexbin + sorted-prediction plots first (before any tables) — the fastest
   visual read on whether a run is even in the right ballpark.
2. Per-bin comparison against the current baseline (`plot_delta_predictions`,
   via `scripts/analyze_model.py --delta-vs`).
3. Training curves, then the validation metrics table.
4. Per-protein ranking — top 5 / bottom 5 / most sensitive — reusing the
   pattern from `notebooks/decisions/07_message_passing_analysis.ipynb`
   (cells 15-17).
5. Decision, based on **validation RMSE, Pearson r, and per-ESP-bin
   improvement** — not aggregate test means alone.

Once a winner is picked, symlink it as the new baseline so later phases
don't need to re-run it for comparison plots, e.g.:

```bash
ln -sfn phase_a/attention_pw05 checkpoints/baseline_attention
ln -sfn phase_a/distance_pw05  checkpoints/baseline_distance
```

Delete this instructional cell once you've copied the template.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path("../..").resolve()))


def show_png_grid(runs, filename, title, ncols=2):
    """Display saved PNG plots from each run's plot_dir in a grid."""
    available = [r for r in runs if (r["plot_dir"] / filename).exists()]
    if not available:
        print(f"No '{filename}' plots found. Run analyze_model.py --save-plots first.")
        return
    nrows = (len(available) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
    axes = axes.flatten() if nrows * ncols > 1 else [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    for ax, run in zip(axes, available):
        ax.imshow(mpimg.imread(run["plot_dir"] / filename))
        ax.set_title(run["label"], fontsize=11)
        ax.axis("off")
    for ax in axes[len(available):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_metric_bars(df, title_prefix, metrics):
    """1-row x 2-col grouped bar chart: one chart per model type.

    metrics: list of (column_name, color) tuples.
    """
    if df.empty:
        print("No data to plot.")
        return
    model_types = ["Attention", "Distance"]
    fig, axes = plt.subplots(1, len(model_types), figsize=(14, 5), sharey=False)
    fig.suptitle(title_prefix, fontsize=13, fontweight="bold")
    n_metrics = len(metrics)
    total_width = 0.7
    bar_w = total_width / n_metrics
    for ax, model_type in zip(axes, model_types):
        sub = df[df["Model"] == model_type].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        x = range(len(sub))
        for i, (metric, color) in enumerate(metrics):
            if metric not in sub.columns or sub[metric].isna().all():
                continue
            offsets = [xi - total_width / 2 + bar_w * i + bar_w / 2 for xi in x]
            bars = ax.bar(offsets, sub[metric], width=bar_w, color=color, label=metric, zorder=3)
            ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=90)
        ax.set_title(model_type, fontsize=12)
        ax.set_xlabel(sub.columns[2] if len(sub.columns) > 2 else "Run")
        ax.set_xticks(list(x))
        ax.set_xticklabels(sub["Run"].tolist(), fontsize=9, rotation=30, ha="right")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3, zorder=0)
    plt.tight_layout()
    plt.show()

## 1. Configuration

In [ ]:
THESIS_ROOT = Path("/home/student/thesis")
CKPT_ROOT   = THESIS_ROOT / "checkpoints"
EVAL_ROOT   = THESIS_ROOT / "model_eval"

PHASE = "phase_a"  # <- set to this phase's folder name

# Baseline symlinks (see Section 7 of the previous winning phase notebook).
BASELINE_CKPT = {
    "attention": CKPT_ROOT / "baseline_attention",
    "distance":  CKPT_ROOT / "baseline_distance",
}

# One dict per run in this phase. `suffix` must match the sweep YAML's
# `suffix:` for that run.
SUFFIXES = ["pw00", "pw03", "pw05", "pw08", "pw10"]

RUNS = [
    dict(label=f"{model.capitalize()} — {suffix}", model_type=model, suffix=suffix,
         plot_dir=EVAL_ROOT/PHASE/f"{model}_{suffix}", ckpt_dir=CKPT_ROOT/PHASE/f"{model}_{suffix}")
    for model in ["attention", "distance"]
    for suffix in SUFFIXES
]

print(f"{'Run':<22}  {'Plots':>6}  {'Metrics':>8}")
print("-" * 42)
for r in RUNS:
    has_plots   = r["plot_dir"].exists()
    has_metrics = (r["ckpt_dir"] / "metrics.csv").exists()
    print(f"{r['label']:<22}  {'yes' if has_plots else 'no':>6}  {'yes' if has_metrics else 'no':>8}")

## 2. Hexbin + Sorted Predictions (top of notebook per the new standard)

Fastest visual read: hexbin shows overall density/compression, sorted-prediction
shows behavior at the ESP extremes.

In [ ]:
show_png_grid(RUNS, "parity_hexbin.png", "Parity Hexbin", ncols=2)

In [ ]:
show_png_grid(RUNS, "sorted_predictions.png", "Sorted Predictions", ncols=2)

## 3. Per-Bin Comparison vs Baseline

Generate with, for each run:

```bash
python scripts/analyze_model.py --checkpoint-dir <run_ckpt_dir> \
    --delta-vs checkpoints/baseline_<model> --save-plots <run_plot_dir>
```

Positive (blue) bars = this run beats the baseline in that ESP bin.

In [ ]:
DELTA_PREFIX = ""  # matches the --model prefix used when generating, if any
show_png_grid(
    [dict(r, plot_dir=r["plot_dir"]) for r in RUNS],
    f"{DELTA_PREFIX}delta_vs_baseline_{{}}.png",  # fill in per-model baseline name if needed
    "Per-Bin Delta vs Baseline",
    ncols=2,
)

## 4. Training Curves

In [ ]:
show_png_grid(RUNS, "training_curves.png", "Training Curves")

## 5. Validation Metrics Comparison (selection basis)

**Winners are chosen from validation metrics, not test metrics** — using test
numbers to pick a phase winner means implicitly fitting the test set across
every phase. Primary metrics: validation RMSE, Pearson r (both at the
`best_model.pt` epoch), and the per-bin delta from Section 3. Test metrics
below are a sanity check only.

In [ ]:
rows = []
for run in RUNS:
    csv_path = run["ckpt_dir"] / "metrics.csv"
    if not csv_path.exists():
        continue
    hist = pd.read_csv(csv_path)
    if hist.empty:
        continue
    best = hist.loc[hist["val_loss"].idxmin()]
    rows.append({
        "Run":           run["label"],
        "Model":         run["model_type"].capitalize(),
        "Pearson r":     best["val_pearson_r"],
        "RMSE":          best["val_rmse"],
        "Val loss":      best["val_loss"],
        "Train loss":    best["train_loss"],
        "Train/val gap": best["train_loss"] - best["val_loss"],
        "Best epoch":    int(best["epoch"]),
    })

val_df = pd.DataFrame(rows).sort_values(["Model", "Pearson r"], ascending=[True, False])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(val_df)
plot_metric_bars(val_df, "Validation metrics (selection basis)",
                  metrics=[("Pearson r", "steelblue"), ("RMSE", "darkorange")])

## 6. Per-Protein Ranking

Reuses the pattern from `notebooks/decisions/07_message_passing_analysis.ipynb`
(cells 15-17): per-protein Pearson r from each run's `test_metrics.json`,
pivoted to a proteins × runs table, then ranked.

In [ ]:
import numpy as np

pp_data: dict[str, dict[str, float]] = {}

for run in RUNS:
    metrics_path = run["ckpt_dir"] / "test_metrics.json"
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        data = json.load(f)
    pp = data.get("per_protein", {})
    pp_data[run["label"]] = {pid: v["pearson_r"] for pid, v in pp.items()}

all_proteins = sorted({pid for d in pp_data.values() for pid in d})
run_labels   = list(pp_data.keys())

pivot = pd.DataFrame(
    {label: [pp_data[label].get(pid, float("nan")) for pid in all_proteins]
     for label in run_labels},
    index=all_proteins,
)

def _short(pid):
    return pid.removeprefix("AF-").removesuffix("-F1")

pivot.index = [_short(p) for p in pivot.index]

pivot["_mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("_mean", ascending=False).drop(columns=["_mean"])

print(f"Loaded per-protein data for {len(pivot)} proteins across {len(run_labels)} runs.")

In [ ]:
rank_df = pivot.rank(ascending=False, method="min").astype(int)

rank_df.insert(0, "Mean r",    pivot.mean(axis=1).round(4))
rank_df.insert(1, "Std r",     pivot.std(axis=1).round(4))
rank_df.insert(2, "Mean rank", rank_df.iloc[:, 2:].mean(axis=1).round(2))
rank_df.insert(3, "Rank std",  rank_df.iloc[:, 3:].std(axis=1).round(2))

rank_df = rank_df.sort_values("Mean rank")

print("Top 5 proteins (lowest mean rank = most consistently well-predicted):")
display(rank_df.head(5))

print("\nBottom 5 proteins (highest mean rank = most consistently difficult):")
display(rank_df.tail(5))

print("\nMost sensitive proteins (highest rank std across configs):")
display(rank_df.sort_values("Rank std", ascending=False).head(5))

## 7. Decision

*Fill in once this phase's runs complete.*

| Model | Winning config | Val Pearson r | Val RMSE | Per-bin verdict (Sec. 3) | Runner-up |
|---|---|---|---|---|---|
| Attention | ? | ? | ? | ? | ? |
| Distance | ? | ? | ? | ? | ? |

**Reasoning:** *why the winner was chosen — especially if it wasn't simply
the top row of Section 5 (e.g. picked for a smaller train/val gap, or a
clearly better per-bin verdict despite a marginal aggregate-metric edge).*

**Carried forward:** *symlink the winner as the new baseline:*

```bash
ln -sfn <phase>/<model>_<winning_suffix> checkpoints/baseline_<model>
```